# 20 · Query Decomposition 与 Step-back

> 有些问题太“复合”，一次检索说不清；有些问题太“深”，直接答会卡在细节。Decomposition 负责**拆**，Step-back 负责**退一步**。

**本文件覆盖知识点**：Query Decomposition / Sub-question Generation / Parallel Retrieval / Sequential Retrieval / Step-back Prompting

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Query Decomposition：拆成子问题

```text
原问: Redis 和 Kafka 分别如何保证消息/数据可靠性？有什么区别？
  Q1: Redis 如何保证数据可靠性？
  Q2: Kafka 如何保证消息可靠性？
  Q3: Redis 与 Kafka 的可靠性机制有何区别？
```

- **Parallel（并行）**：子问题相互独立 → 同时检索，答案分头取；
- **Sequential（串行）**：子问题有依赖（先答 A 才能答 B）→ 前一步答案参与下一步（也是 Agent 化雏形）。

> 判定何时拆：先让 LLM 判断“该问题是否需要多步知识”。这也是 **Adaptive RAG（第 29 课）** 的路由思想。

In [ ]:
# .env 配置
from dotenv import load_dotenv; load_dotenv()
import os, json
from dashscope import Generation
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def decompose(question):
    """把复合问题拆成独立子问题列表"""
    prompt = (f'把问题拆成 2-4 个相互独立、能单独检索的子问题，输出 JSON 字符串数组。\n问题: {question}')
    r = Generation.call(model='qwen-plus', messages=[{'role':'user','content':prompt}],
                        api_key=API_KEY, result_format='message')
    raw = r.output.choices[0].message.content.strip().strip('`')
    if raw.startswith('json'): raw = raw[4:].strip()
    try:
        return json.loads(raw)
    except Exception:
        return [question]

if API_KEY and '你的' not in API_KEY:
    qs = decompose('Redis 与 Kafka 如何保证可靠性？二者有何区别？')
    for i, q in enumerate(qs): print(f'Q{i+1}: {q}')

In [ ]:
# 知识点·真调说明：何时该拆（Decomposition 前置判定）—— 让模型先判断“需不需要多步知识”，而不是逢问必拆
import json as _json
_fall = ('[{"question": "Redis 支持哪些数据结构？", "need_split": false, "reason": "单点事实型，一次检索即可"}, '
         '{"question": "Redis 与 Kafka 如何保证数据/消息可靠性？二者有何区别？", "need_split": true, "reason": "含两个主体加对比，需拆成子问题"}, '
         '{"question": "Kafka 的消费者组是什么？", "need_split": false, "reason": "单一概念，一次检索即可"}]')
out = _llm_live(
    prompt="""判断下面每个问题是否需要“拆成多个子问题再各自检索”，输出 JSON 数组，不解释。
问题1：Redis 支持哪些数据结构？
问题2：Redis 与 Kafka 如何保证数据/消息可靠性？二者有何区别？
问题3：Kafka 的消费者组是什么？

JSON 格式：[{"question": 问题原文, "need_split": true 或 false, "reason": 一句话理由}]""",
    system='你是 RAG 的拆解判定器：事实型/单点问题 need_split=false；含多主体、多层依赖或对比的问题 need_split=true。',
    fallback=_fall,
    temperature=0.1,
)
s = out if out is not None else _fall
s = s.strip().strip('`')
if s.startswith('json'):
    s = s[4:].strip()
if out is None:
    print('（以上为固定样例；下面用样例走 json.loads 解析）')
try:
    for it in _json.loads(s):
        if it['need_split']:
            mark = '需要拆解 → 走并行/串行子检索'
        else:
            mark = '一次检索即可，不必拆'
        print('• %s（%s）%s' % (it['question'], it['reason'], mark))
except Exception as e:
    print('未通过 json.loads：', e)
print('→ 先判定“要不要拆”再决定检索策略，就是 Adaptive RAG 的路由思想（29 课深入）；避免把简单问题白白拆成多路延迟与成本。')

## 2. Step-back Prompting：先退一步再作答

知识型难题直接检索可能“只见树木不见森林”。先退到**抽象原则层**：

```text
问题: 为什么这个 MySQL SQL 用不上索引？
退一步: MySQL 索引失效的常见原因有哪些？(原则)
再回来: 该 SQL 属于哪种失效场景？          (套用)
```

Step-back 常与 **HyDE**（第 39 课先写假设答案）混用：退一步得到的“原则性答案”本身就能当检索的 Query。



In [ ]:
# 知识点·真调说明：Step-back Prompting —— 同一道原理题，「直接答」vs「先退到原则层再套回」，看回答的结构与覆盖差别
print('① 直接回答（见题答题，不后退）')
_llm_live(
    prompt="""为什么 MySQL 里这条 SQL 用不上 create_time 上的索引？
SELECT * FROM orders WHERE status = "PENDING" AND create_time > "2026-01-01";""",
    system='你是 MySQL 优化讲师，直接针对这条 SQL 回答原因。',
    fallback="""这条 SQL 的 create_time > "2026-01-01" 本身是合法范围条件，未必破坏索引；
若仍没走 create_time 索引，更可能是 status 等值过滤的选择性更高、优化器选了 status 索引，
或表数据量小导致成本估算选择全表扫描。""",
    temperature=0.2,
)
print()
print('② Step-back：先退到抽象原则层列出“索引失效常见场景”，再回到这条具体 SQL 逐条排查')
_llm_live(
    prompt="""为什么 MySQL 里这条 SQL 用不上 create_time 上的索引？
SELECT * FROM orders WHERE status = "PENDING" AND create_time > "2026-01-01";""",
    system="""你是 MySQL 优化讲师，请严格分两步作答：
第一步先退到抽象原则：列出“MySQL 索引失效的常见原因”（对列套函数、隐式类型转换、LIKE 前导通配、OR 连接非索引列、范围+排序组合、优化器成本估算认为走索引更慢等）；
第二步再回到这条具体 SQL，对照上述原则逐条排查，判断它属于哪种场景并给出最终结论。""",
    fallback="""第一步（原则）：索引失效常见有 ①对列套函数/运算 ②隐式类型转换 ③前导通配 LIKE ④OR 连接非索引列 ⑤复合索引未用最左前缀 ⑥优化器估算走索引更慢。
第二步（套回）：本 SQL 的 create_time > "2026-01-01" 是合法范围条件、未破坏索引；若仍未走索引，
多半是 status 等值过滤选择性高（优化器选 status 索引）或数据量小被估成全表扫描——属第⑥类。""",
    temperature=0.2,
)
print()
print('对比：② 先给“原则清单”再“套用”，答得更结构化、不易漏排查项。')
print('→ Step-back 退出来的“原则性答案”本身就能当检索 Query（HyDE 思路，39 课深入）；先喂原则再回来答，是答深原理题的通用手法。')

## 小结

- 复合问题用 **Decomposition**（并/串行召回）；
- 深层原理问题用 **Step-back** 退到原则；
- 两者都在“query 形态”上做文章，是 Agentic RAG 规划的萌芽。